# CVaR (padronizado)


## 1) Importação do dataset e bibliotecas


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

CSV_PATH = "../../data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15


## 2) Execução do `setup()` e alinhamento temporal


In [ ]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    save_artifacts=False,
)

processed_df = raw_df.reset_index().copy()
if "Change %" in processed_df.columns:
    processed_df["Change %"] = (
        processed_df["Change %"].str.replace("%", "", regex=False).astype(float) / 100
    )

processed_df = processed_df.sort_values("Date")
processed_df["target_return"] = (
    processed_df[TARGET_COL].pct_change(HORIZON).shift(-HORIZON)
)
processed_df = processed_df.dropna().set_index("Date")

n_rows = len(processed_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
val_index = processed_df.iloc[train_end:val_end].index

print("Validation length:", len(val_index))


## 3) Construção da ferramenta de risco (CVaR rolling)


In [ ]:
returns = processed_df[TARGET_COL].pct_change().fillna(0.0)

def cvar_from_window(window_values: np.ndarray, alpha: float) -> float:
    q = np.quantile(window_values, alpha)
    tail = window_values[window_values <= q]
    if tail.size == 0:
        return float(q)
    return float(tail.mean())

rolling_window = 100
cvar_95_full = returns.rolling(rolling_window, min_periods=20).apply(
    lambda x: cvar_from_window(x.values, alpha=0.05),
    raw=False,
)
cvar_99_full = returns.rolling(rolling_window, min_periods=20).apply(
    lambda x: cvar_from_window(x.values, alpha=0.01),
    raw=False,
)

tail_loss_expectation_full = (0.5 * cvar_95_full) + (0.5 * cvar_99_full)

cvar_dataset = pd.DataFrame(index=val_index)
cvar_dataset["cvar_95"] = cvar_95_full.reindex(val_index)
cvar_dataset["cvar_99"] = cvar_99_full.reindex(val_index)
cvar_dataset["tail_loss_expectation"] = tail_loss_expectation_full.reindex(val_index)

cvar_dataset = cvar_dataset.bfill().sort_index()


## 4) Sanity checks mínimos


In [ ]:
print("NaN críticos:")
print(cvar_dataset.isna().sum())

print("\nDistribuição plausível:")
print(cvar_dataset.describe().T[["mean", "std", "min", "max"]])

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", cvar_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", cvar_dataset.index.has_duplicates)

print("Dataset lines == val_index lines:", len(cvar_dataset) == len(val_index))
print("Dataset index == val_index:", cvar_dataset.index.equals(val_index))


## 5) DataFrame final (features de risco) e 6) Padronização


In [ ]:
cvar_dataset = cvar_dataset[["cvar_95", "cvar_99", "tail_loss_expectation"]]

cvar_dataset.head(), cvar_dataset.shape


## 7) Salvamento (opcional)


In [ ]:
# cvar_dataset.to_parquet("cvar_dataset.parquet", index=True)
